# Assessment 1 - Source-to-Bronze Profiling & Reconciliation

See `docs/milestones.md` and `docs/design/assignment.md` for task scope.
Connectivity conventions: see `00_template_connectivity_check.ipynb`.


In [1]:
import os
from pyspark.sql import SparkSession
import psycopg2

POSTGRES_DB = os.environ["POSTGRES_DB"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_PASSWORD = os.environ["POSTGRES_PASSWORD"]


## Task 1 - Data Profiling

See `results/assessment-1/assessment-1-overview.md` for scenario, table shapes, and scale.

This section implements **09.CK.01** (task ref `01.01` - record/distinct counts) only.
Remaining task 1 checks (09.CK.02-10) are scoped for a later pass.


In [2]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment1-task1-profiling")
    .getOrCreate()
)


def jdbc_table(table_name):
    return spark.read.jdbc(
        url=f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
        table=table_name,
        properties={
            "user": POSTGRES_USER,
            "password": POSTGRES_PASSWORD,
            "driver": "org.postgresql.Driver",
        },
    )


src_df = jdbc_table("src_transaction_daily")
bronze_df = jdbc_table("bronze.transaction_daily")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/30 12:15:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.functions import countDistinct


def record_distinct_counts(df, label):
    record_count = df.count()
    distinct_count = df.select(countDistinct("transaction_id")).collect()[0][0]
    gap = record_count - distinct_count
    status = "PASS" if record_count > 0 and distinct_count > 0 else "FAIL"
    print(
        f"[{status}] 09.CK.01 {label}: "
        f"record_count={record_count} distinct_count={distinct_count} gap={gap}"
    )
    return record_count, distinct_count


src_record_count, src_distinct_count = record_distinct_counts(src_df, "src_transaction_daily")
bronze_record_count, bronze_distinct_count = record_distinct_counts(
    bronze_df, "bronze.transaction_daily"
)

check_01_01_status = (
    "PASS" if min(src_record_count, bronze_record_count) > 0 else "FAIL"
)
print(f"[{check_01_01_status}] assessment1-profiling-09.CK.01: overall status={check_01_01_status}")

spark.stop()


[PASS] 09.CK.01 src_transaction_daily: record_count=2010 distinct_count=2000 gap=10


[PASS] 09.CK.01 bronze.transaction_daily: record_count=1993 distinct_count=1975 gap=18
[PASS] assessment1-profiling-09.CK.01: overall status=PASS
